# Performance- und Machbarkeitstest: A*-Router auf großem Netzwerk

Dieses Notebook evaluiert die Leistung, Lösbarkeit und Konsolidierungsrate des `AStarRouter` auf dem großen Netzwerk (`large_network.json`) mit einem Planungshorizont von 30 Tagen und verschiedenen Sendungsmengen (1, 2, 5, 10, 50, 100, 500, 1000, 5000, 10000, 20000).

Jede Sendung besitzt ihre eigene, zufällig generierte Gewichtung der Ziele (Kosten, Zeit und Emissionen).

Die Berechnungsergebnisse werden zwischengespeichert, um eine Fortführung nach einer Unterbrechung zu ermöglichen.

## 1. Setup und Imports

In [5]:
import sys
import time
import random
from pathlib import Path
from collections import defaultdict
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from freight_routing.data_loader import NetworkDataLoader
from freight_routing.data_models import Shipment, ObjectiveWeights, ArcType
from freight_routing.model import TimeExpandedNetwork
from heuristics.dijkstra_router import AStarRouter

## 2. Laden des Netzwerks

In [6]:
network_path = PROJECT_ROOT / "dataset/large_network.json"
print(f"Lade Netzwerk aus {network_path}...")
t0 = time.time()
network_data = NetworkDataLoader.from_json(network_path)
print(f"Netzwerk in {time.time() - t0:.2f} s geladen.")
network_data.summary()

Lade Netzwerk aus /home/phil/studium/4Semester/or/Sustainable_Freight_Mode_Choice/dataset/large_network.json...
Netzwerk in 0.23 s geladen.
Summary NetworkData:
hubs=870
arcs=36272
modes=4


## 3. Generierung von 20.000 Sendungen mit individuellen Gewichtungen

In [7]:
# Seed für Reproduzierbarkeit
random.seed(42)
N_SHIPMENTS = 20000
PLANNING_DAYS = 30
DEADLINE = PLANNING_DAYS * 24 * 60  # 30 Tage in Minuten (43.200 min)

hubs_list = list(network_data.hubs.keys())
shipments = []
for i in range(N_SHIPMENTS):
    start = random.choice(hubs_list)
    dest = random.choice(hubs_list)
    while dest == start:
        dest = random.choice(hubs_list)

    # Zufällige Gewichtungen generieren, die sich zu 1.0 aufsummieren
    w_cost = random.uniform(0.01, 1.0)
    w_time = random.uniform(0.01, 1.0)
    w_emissions = random.uniform(0.01, 1.0)
    total_w = w_cost + w_time + w_emissions
    w_cost /= total_w
    w_time /= total_w
    w_emissions /= total_w

    shipments.append(
        Shipment(
            id=f"shipment_{i}",
            start_hub=start,
            end_hub=dest,
            start_time=0,
            deadline=DEADLINE,
            max_price=1000000.0,
            max_emissions=None,
            weight=float(random.randint(1, 10)),
            objective_weights=ObjectiveWeights(
                cost=w_cost,
                time=w_time,
                emissions=w_emissions
            )
        )
    )
print(f"{len(shipments)} Sendungen mit zufälligen Gewichtungen generiert.")

20000 Sendungen mit zufälligen Gewichtungen generiert.


## 4. Durchführung des Benchmarks mit Zwischenspeicherung (Caching)

Wir führen das Routing für verschiedene Sendungsmengen durch. Wenn die CSV-Datei `dataset/dijkstra_performance_results.csv` bereits existiert, werden bereits berechnete Größen übersprungen.

In [ ]:
def analyze_result(res, duration, total_tested):
    solved = len(res.shipment_routes)
    unsolvable = total_tested - solved
    pct_unsolvable = (unsolvable / total_tested) * 100
    avg_time = duration / total_tested
    
    # Bündelungs-Analyse
    transport_arc_shipments = defaultdict(list)
    for s_id, route in res.shipment_routes.items():
        for arc in route:
            if arc.arc_type == ArcType.TRANSPORT:
                transport_arc_shipments[arc].append(s_id)

    consolidated_shipments = set()
    for arc, s_ids in transport_arc_shipments.items():
        if len(s_ids) >= 2:
            for s_id in s_ids:
                consolidated_shipments.add(s_id)

    num_consolidated = len(consolidated_shipments)
    pct_consolidated = (num_consolidated / solved) * 100 if solved > 0 else 0.0
    
    return {
        "solved": solved,
        "unsolvable": unsolvable,
        "pct_unsolvable": pct_unsolvable,
        "avg_time": avg_time,
        "num_consolidated": num_consolidated,
        "pct_consolidated": pct_consolidated,
        "obj_val": res.objective_value
    }

results_csv = PROJECT_ROOT / "dataset/dijkstra_performance_results.csv"
sizes_to_test = [1, 2, 5, 10, 50, 100, 500, 1000, 5000, 10000, 20000]

# Lade existierende Ergebnisse oder erstelle leeres DataFrame
if results_csv.exists():
    df_results = pd.read_csv(results_csv)
    completed_sizes = set(df_results["num_shipments"].tolist())
    print(f"Bisher berechnete Sendungsmengen geladen: {completed_sizes}")
else:
    df_results = pd.DataFrame(columns=[
        "num_shipments", "routing_duration", "avg_time_per_shipment",
        "solved_count", "unsolvable_count", "pct_unsolvable",
        "num_consolidated", "pct_consolidated", "objective_value"
    ])
    completed_sizes = set()

weights_default = ObjectiveWeights(cost=0.4, emissions=0.4, time=0.2)

for size in sizes_to_test:
    if size in completed_sizes:
        print(f"Größe {size} bereits berechnet. Überspringe...")
        continue
        
    print(f"\n--- Starte Berechnung für {size} Sendungen ---")
    test_shipments = shipments[:size]
    
    # Netzwerk bauen
    t_build_start = time.time()
    network = TimeExpandedNetwork.build(network_data, planning_days=PLANNING_DAYS, shipments=test_shipments)
    t_build_duration = time.time() - t_build_start
    print(f"Netzwerk gebaut in {t_build_duration:.2f} s")
    
    # Routing berechnen
    router = AStarRouter(objective_weights=weights_default)
    t_routing_start = time.time()
    routing_result = router.solve_multiple(network, show_progress=True)
    t_routing_duration = time.time() - t_routing_start
    print(f"Routing berechnet in {t_routing_duration:.2f} s")
    
    # Ergebnisse analysieren
    stats = analyze_result(routing_result, t_routing_duration, size)
    
    # In DataFrame eintragen und speichern
    new_row = pd.DataFrame([{
        "num_shipments": size,
        "routing_duration": t_routing_duration,
        "avg_time_per_shipment": stats["avg_time"],
        "solved_count": stats["solved"],
        "unsolvable_count": stats["unsolvable"],
        "pct_unsolvable": stats["pct_unsolvable"],
        "num_consolidated": stats["num_consolidated"],
        "pct_consolidated": stats["pct_consolidated"],
        "objective_value": stats["obj_val"]
    }])
    
    df_results = pd.concat([df_results, new_row], ignore_index=True)
    # Sortieren nach Sendungsmenge vor dem Abspeichern
    df_results = df_results.sort_values("num_shipments").reset_index(drop=True)
    df_results.to_csv(results_csv, index=False)
    print(f"Ergebnisse für {size} Sendungen in {results_csv} gespeichert.")

Bisher berechnete Sendungsmengen geladen: {1, 2, 5}
Größe 1 bereits berechnet. Überspringe...
Größe 2 bereits berechnet. Überspringe...
Größe 5 bereits berechnet. Überspringe...

--- Starte Berechnung für 10 Sendungen ---
Netzwerk gebaut in 31.46 s


Routing shipments: 100%|██████████| 10/10 [00:11<00:00,  1.17s/it]


Routing berechnet in 13.81 s
Ergebnisse für 10 Sendungen in /home/phil/studium/4Semester/or/Sustainable_Freight_Mode_Choice/dataset/dijkstra_performance_results.csv gespeichert.

--- Starte Berechnung für 50 Sendungen ---
Netzwerk gebaut in 29.17 s


Routing shipments: 100%|██████████| 50/50 [00:17<00:00,  2.81it/s]


Routing berechnet in 20.50 s
Ergebnisse für 50 Sendungen in /home/phil/studium/4Semester/or/Sustainable_Freight_Mode_Choice/dataset/dijkstra_performance_results.csv gespeichert.

--- Starte Berechnung für 100 Sendungen ---
Netzwerk gebaut in 29.77 s


...

Routing shipments: 100%|██████████| 500/500 [01:09<00:00,  7.18it/s]


Routing berechnet in 80.69 s
Ergebnisse für 500 Sendungen in /home/phil/studium/4Semester/or/Sustainable_Freight_Mode_Choice/dataset/dijkstra_performance_results.csv gespeichert.

--- Starte Berechnung für 1000 Sendungen ---
Netzwerk gebaut in 28.88 s


Routing shipments: 100%|██████████| 1000/1000 [04:27<00:00,  3.74it/s]


Routing berechnet in 287.07 s
Ergebnisse für 1000 Sendungen in /home/phil/studium/4Semester/or/Sustainable_Freight_Mode_Choice/dataset/dijkstra_performance_results.csv gespeichert.

--- Starte Berechnung für 5000 Sendungen ---
Netzwerk gebaut in 30.46 s


Routing shipments:  38%|███▊      | 1902/5000 [22:05<55:56,  1.08s/it]  

## 5. Visualisierung mit Seaborn

In [ ]:
if results_csv.exists():
    df_plot = pd.read_csv(results_csv).sort_values("num_shipments")
    
    # Theme-Einstellungen
    sns.set_theme(style="whitegrid", palette="muted")
    fig, axes = plt.subplots(3, 1, figsize=(10, 15), sharex=False)
    
    # 1. Durchschnittliche Dauer pro Sendung
    sns.lineplot(
        data=df_plot, x="num_shipments", y="avg_time_per_shipment",
        marker="o", color="b", linewidth=2.5, ax=axes[0]
    )
    axes[0].set_xscale("log")
    axes[0].set_title("A*-Router: Durchschnittliche Laufzeit pro Sendung", fontsize=14, fontweight="bold")
    axes[0].set_xlabel("Anzahl Sendungen (logarithmische Skala)", fontsize=11)
    axes[0].set_ylabel("Berechnungsdauer pro Sendung (s)", fontsize=11)
    
    # 2. Konsolidierungsrate
    sns.lineplot(
        data=df_plot, x="num_shipments", y="pct_consolidated",
        marker="s", color="g", linewidth=2.5, ax=axes[1]
    )
    axes[1].set_xscale("log")
    axes[1].set_title("Konsolidierungsrate in Abhängigkeit von der Sendungsmenge", fontsize=14, fontweight="bold")
    axes[1].set_xlabel("Anzahl Sendungen (logarithmische Skala)", fontsize=11)
    axes[1].set_ylabel("Konsolidierte Sendungen (%)", fontsize=11)
    axes[1].set_ylim(-5, 105)
    
    # 3. Unlösbare Sendungen
    sns.lineplot(
        data=df_plot, x="num_shipments", y="pct_unsolvable",
        marker="d", color="r", linewidth=2.5, ax=axes[2]
    )
    axes[2].set_xscale("log")
    axes[2].set_title("Anteil unlösbarer Sendungen (Feasibility-Rate)", fontsize=14, fontweight="bold")
    axes[2].set_xlabel("Anzahl Sendungen (logarithmische Skala)", fontsize=11)
    axes[2].set_ylabel("Unlösbare Sendungen (%)", fontsize=11)
    axes[2].set_ylim(-5, 105)
    
    plt.tight_layout()
    plot_img_path = PROJECT_ROOT / "dataset/dijkstra_performance_plots.png"
    plt.savefig(plot_img_path, dpi=300)
    print(f"Grafiken erfolgreich unter {plot_img_path} abgespeichert.")
    plt.show()
else:
    print("Keine Benchmark-Daten zur Visualisierung gefunden. Führen Sie erst den Benchmark aus.")